# VFL Empire — Full Fixture 90%+ Prediction Engine

**Lord FaithDavid's Empire Training Notebook**

This notebook trains per-market classifiers on rich features:
- League table: rank, points, form, GF/GA rates
- H2H history: win rates, goal patterns per market
- Odds: implied probabilities per market
- Season regime: avg goals, over/under rates

**Targets (separate model per market):**
- Over 1.5 Goals | Under 3.5 Goals | GG (Both Score)
- Home Win | Draw | Away Win

**GPU Training:** XGBoost GPU + LightGBM GPU + Neural Net → Ensemble


In [ ]:
# ── OPTIONAL: REMOTE IDE SSH TUNNEL (CLOUDFLARE) ───────────────────────────
# Run this cell to set up an SSH server and access this GPU runtime from VS Code
PASSWORD = "vfl-empire-pass" # Change this password if you want

print("Installing SSH server...")
!apt-get update -qq && apt-get install -y openssh-server > /dev/null
!mkdir -p /var/run/sshd
!echo "root:{PASSWORD}" | chpasswd
!echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
!echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config

# Start SSH service
import subprocess
subprocess.Popen(["/usr/sbin/sshd", "-D"])

print("Downloading Cloudflare tunnel...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# Start Cloudflare tunnel in the background to forward SSH port 22
import os
os.system("cloudflared tunnel --url tcp://localhost:22 > /colab_tunnel.log 2>&1 &")

# Retrieve the Cloudflare tunnel URL
import time, re
time.sleep(5)
with open("/colab_tunnel.log") as f:
    log_content = f.read()
urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
if urls:
    ssh_url = urls[0].replace("https://", "")
    print("=" * 60)
    print("🚀 SSH TUNNEL CONNECT COMMAND:")
    print(f"ssh -o StrictHostKeyChecking=no root@{ssh_url}")
    print(f"Password: {PASSWORD}")
    print("=" * 60)
else:
    print("Could not find Cloudflare URL yet. Printing logs:")
    print(log_content)


In [ ]:
# ── 1. GPU CHECK ─────────────────────────────────────────────────────────────
import torch, psutil, os, json, warnings
warnings.filterwarnings('ignore')

print('=' * 60)
print('VFL EMPIRE — TRAINING ENVIRONMENT')
print('=' * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'GPU Memory: {props.total_memory / 1e9:.1f} GB')
    print(f'CUDA: {torch.version.cuda}')
    USE_GPU = True
else:
    print('WARNING: No GPU — using CPU (slower)')
    USE_GPU = False
print(f'CPU: {psutil.cpu_count()} cores | RAM: {psutil.virtual_memory().total/1e9:.1f} GB')
print('=' * 60)


In [ ]:
# ── 2. INSTALL DEPS ───────────────────────────────────────────────────────────
!pip install -q xgboost lightgbm catboost scikit-learn pandas numpy matplotlib seaborn
print('Packages installed.')


In [ ]:
# ── 3. MOUNT DRIVE & LOAD DATA ────────────────────────────────────────────────
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
DATA_PATH = DRIVE / 'vfl_training_data' / 'vfl_rich_features.csv'
MODELS_DIR = DRIVE / 'vfl_empire_models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Fall back to older training data if rich features not yet exported
if not DATA_PATH.exists():
    DATA_PATH = DRIVE / 'vfl_training_data.csv'
    print(f'Rich features not found — using legacy: {DATA_PATH}')
else:
    print(f'Loading rich features: {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {len(df):,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns[:10])}...')


In [ ]:
# ── 4. FEATURE SELECTION ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# Features available in both rich and legacy datasets
RICH_FEATURES = [
    # League table — home
    'home_rank', 'home_points', 'home_played',
    'home_goals_per_game', 'home_goals_against_per_game', 'home_goal_diff_per_game',
    'home_win_rate', 'home_draw_rate', 'home_form_score', 'home_points_per_game',
    # League table — away
    'away_rank', 'away_points', 'away_played',
    'away_goals_per_game', 'away_goals_against_per_game', 'away_goal_diff_per_game',
    'away_win_rate', 'away_draw_rate', 'away_form_score', 'away_points_per_game',
    # Differentials
    'rank_diff', 'points_diff', 'form_diff', 'goals_diff', 'expected_total_goals',
    # H2H
    'h2h_count', 'h2h_home_win_rate', 'h2h_draw_rate', 'h2h_away_win_rate',
    'h2h_avg_goals', 'h2h_std_goals',
    'h2h_over_15_rate', 'h2h_over_25_rate', 'h2h_over_35_rate',
    'h2h_under_25_rate', 'h2h_under_35_rate',
    'h2h_gg_rate', 'h2h_ng_rate', 'h2h_data_quality',
    # Season regime
    'season_avg_goals', 'season_over_15_rate', 'season_over_25_rate',
    'season_over_35_rate', 'season_under_35_rate', 'season_gg_rate',
    # Odds / implied probs
    'odds_home', 'odds_draw', 'odds_away',
    'odds_over_15', 'odds_under_35', 'odds_over_25', 'odds_gg',
    'impl_home', 'impl_draw', 'impl_away',
    'impl_over_15', 'impl_under_35', 'impl_over_25', 'impl_gg',
    # Context
    'matchday',
]

LEGACY_FEATURES = [
    'confidence', 'odds', 'cv_1x2',
    'prediction_enc', 'engine_enc', 'tier_home_enc', 'tier_away_enc',
    'match_day', 'is_home_win', 'is_away_win', 'is_draw',
    'is_over', 'is_under', 'is_dnb', 'expected_value', 'high_conf', 'very_high_conf'
]

# Detect which features are available
avail_rich = [f for f in RICH_FEATURES if f in df.columns]
avail_legacy = [f for f in LEGACY_FEATURES if f in df.columns]
FEATURES = avail_rich if len(avail_rich) > 10 else avail_legacy

print(f'Using {len(FEATURES)} features ({"rich" if len(avail_rich) > 10 else "legacy"} mode)')
print(f'Features: {FEATURES[:8]}...')

# Markets and their target columns
MARKETS = {
    'over_15': 'target_over_15',
    'under_35': 'target_under_35',
    'gg': 'target_gg',
    'home_win': 'target_home_win',
    'draw': 'target_draw',
    'away_win': 'target_away_win',
    'over_25': 'target_over_25',
    'under_25': 'target_under_25',
}

# Check which markets have target columns
avail_markets = {k: v for k, v in MARKETS.items() if v in df.columns}
print(f'Available markets: {list(avail_markets.keys())}')


In [ ]:
# ── 5. TRAIN PER-MARKET MODELS ────────────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
import pickle, json
from datetime import datetime

def train_market_model(df, features, target_col, market_name, use_gpu=USE_GPU):
    """Train XGBoost + LightGBM ensemble for a single market target."""
    print(f'\n{'='*60}')
    print(f'Training: {market_name.upper()} → {target_col}')
    print(f'{'='*60}')
    
    # Filter to rows with this target
    mask = df[target_col].notna() & df[features].notna().all(axis=1)
    X = df.loc[mask, features].fillna(0).values
    y = df.loc[mask, target_col].values.astype(int)
    
    if len(X) < 500:
        print(f'  Skipping — only {len(X)} rows')
        return None
    
    print(f'  Rows: {len(X):,} | Positive rate: {y.mean():.1%}')
    
    # Train/validation split (time-ordered — use last 20% for validation)
    split = int(len(X) * 0.80)
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]
    
    # ── XGBoost ──────────────────────────────────────────────────────────────
    xgb_params = {
        'n_estimators': 800,
        'max_depth': 6,
        'learning_rate': 0.03,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 5,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'use_label_encoder': False,
        'eval_metric': 'logloss',
        'early_stopping_rounds': 30,
        'verbosity': 0,
    }
    if use_gpu:
        xgb_params['tree_method'] = 'hist'
        xgb_params['device'] = 'cuda'
    
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    xgb_prob_val = xgb_model.predict_proba(X_val)[:, 1]
    xgb_acc = accuracy_score(y_val, xgb_prob_val >= 0.5)
    xgb_auc = roc_auc_score(y_val, xgb_prob_val)
    print(f'  XGBoost:   Acc={xgb_acc:.3f} | AUC={xgb_auc:.3f}')
    
    # ── LightGBM ─────────────────────────────────────────────────────────────
    lgb_params = {
        'n_estimators': 800,
        'max_depth': 6,
        'learning_rate': 0.03,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_samples': 20,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'verbose': -1,
        'early_stopping_rounds': 30,
    }
    if use_gpu:
        lgb_params['device'] = 'gpu'
    
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
    )
    lgb_prob_val = lgb_model.predict_proba(X_val)[:, 1]
    lgb_acc = accuracy_score(y_val, lgb_prob_val >= 0.5)
    lgb_auc = roc_auc_score(y_val, lgb_prob_val)
    print(f'  LightGBM:  Acc={lgb_acc:.3f} | AUC={lgb_auc:.3f}')
    
    # ── Ensemble ─────────────────────────────────────────────────────────────
    ensemble_prob = (xgb_prob_val + lgb_prob_val) / 2
    ens_acc = accuracy_score(y_val, ensemble_prob >= 0.5)
    ens_auc = roc_auc_score(y_val, ensemble_prob)
    ens_brier = brier_score_loss(y_val, ensemble_prob)
    print(f'  Ensemble:  Acc={ens_acc:.3f} | AUC={ens_auc:.3f} | Brier={ens_brier:.3f}')
    
    # ── Isotonic Calibration ──────────────────────────────────────────────────
    # Calibrate so predicted 90% ≈ actual 90% win rate
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(ensemble_prob, y_val)
    cal_prob = iso.transform(ensemble_prob)
    print(f'  Calibrated: Acc={accuracy_score(y_val, cal_prob >= 0.5):.3f}')
    
    # ── Confidence bucket analysis ────────────────────────────────────────────
    print(f'  Confidence buckets (calibrated):')
    for lo, hi in [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.0)]:
        mask_b = (cal_prob >= lo) & (cal_prob < hi)
        n_b = mask_b.sum()
        if n_b > 10:
            acc_b = y_val[mask_b].mean()
            print(f'    {lo:.0%}-{hi:.0%}: n={n_b:4d} actual_win={acc_b:.1%}')
    
    return {
        'xgb': xgb_model,
        'lgb': lgb_model,
        'iso': iso,
        'features': features,
        'market': market_name,
        'target': target_col,
        'val_acc': ens_acc,
        'val_auc': ens_auc,
        'val_brier': ens_brier,
        'train_rows': len(X_train),
        'val_rows': len(X_val),
        'positive_rate': float(y.mean()),
        'trained_at': datetime.now().isoformat(),
    }

# Train all markets
models = {}
for market_name, target_col in avail_markets.items():
    result = train_market_model(df, FEATURES, target_col, market_name)
    if result:
        models[market_name] = result

print(f'\n✅ Trained {len(models)} market models')


In [ ]:
# ── 6. SAVE MODELS TO DRIVE ───────────────────────────────────────────────────
import pickle, json
from pathlib import Path

MODELS_DIR = Path('/content/drive/MyDrive/vfl_empire_models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

summary = {}
for market_name, model_data in models.items():
    # Save XGBoost model
    xgb_path = MODELS_DIR / f'xgb_{market_name}.json'
    model_data['xgb'].save_model(str(xgb_path))
    
    # Save LightGBM model
    lgb_path = MODELS_DIR / f'lgb_{market_name}.txt'
    model_data['lgb'].booster_.save_model(str(lgb_path))
    
    # Save isotonic calibrator
    iso_path = MODELS_DIR / f'iso_{market_name}.pkl'
    with open(iso_path, 'wb') as f:
        pickle.dump(model_data['iso'], f)
    
    summary[market_name] = {
        'val_acc': round(model_data['val_acc'], 4),
        'val_auc': round(model_data['val_auc'], 4),
        'val_brier': round(model_data['val_brier'], 4),
        'train_rows': model_data['train_rows'],
        'positive_rate': model_data['positive_rate'],
        'trained_at': model_data['trained_at'],
        'xgb_path': str(xgb_path),
        'lgb_path': str(lgb_path),
        'iso_path': str(iso_path),
    }

# Save features list
features_info = {'features': FEATURES, 'markets': list(models.keys())}
with open(MODELS_DIR / 'model_features.json', 'w') as f:
    json.dump(features_info, f, indent=2)

# Save training summary
with open(MODELS_DIR / 'training_summary_rich.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n=== TRAINING SUMMARY ===')
for market, stats in summary.items():
    print(f'  {market:12s}: Acc={stats["val_acc"]:.3f} | AUC={stats["val_auc"]:.3f} | rows={stats["train_rows"]:,}')

print(f'\nAll models saved to: {MODELS_DIR}')


In [ ]:
# ── 7. BACKTEST — SIMULATE ALL 8 FIXTURES PER MATCHDAY ───────────────────────
import pandas as pd
import numpy as np
from sklearn.isotonic import IsotonicRegression

print('\n=== BACKTEST: FULL FIXTURE COVERAGE ===')
print('How many fixtures can we predict per matchday at various confidence thresholds?\n')

def predict_fixture(row, models, features):
    """Score all markets for one fixture row. Returns dict of market → (prob, conf_pct)."""
    X = pd.DataFrame([row[features].fillna(0)])
    results = {}
    for market_name, model_data in models.items():
        xgb_p = float(model_data['xgb'].predict_proba(X.values)[0, 1])
        lgb_p = float(model_data['lgb'].predict_proba(X.values)[0, 1])
        ensemble_p = (xgb_p + lgb_p) / 2
        cal_p = float(model_data['iso'].transform([ensemble_p])[0])
        results[market_name] = {
            'prob': round(cal_p, 3),
            'confidence': int(cal_p * 100),
            'bet': cal_p >= 0.75,
        }
    return results

# Use validation set rows (last 20%)
split_idx = int(len(df) * 0.80)
df_val = df.iloc[split_idx:].copy()

if 'home_team' in df_val.columns and 'away_team' in df_val.columns:
    # Group by matchday
    matchday_col = 'matchday' if 'matchday' in df_val.columns else 'match_day'
    groupby_cols = ['season_name', matchday_col] if 'season_name' in df_val.columns else [matchday_col]
    
    results_by_threshold = {0.65: [], 0.70: [], 0.75: [], 0.80: [], 0.85: [], 0.90: []}
    
    for group_key, group in df_val.groupby(groupby_cols):
        fixtures_predicted = {t: 0 for t in results_by_threshold}
        total_fixtures = len(group)
        
        for _, row in group.iterrows():
            preds = predict_fixture(row, models, FEATURES)
            best_conf = max(p['prob'] for p in preds.values())
            for threshold in results_by_threshold:
                if best_conf >= threshold:
                    fixtures_predicted[threshold] += 1
        
        for t in results_by_threshold:
            results_by_threshold[t].append(fixtures_predicted[t] / max(total_fixtures, 1))
    
    print('Average fixtures predicted per matchday (out of 8):')
    for threshold, coverages in results_by_threshold.items():
        avg_cov = np.mean(coverages)
        avg_count = avg_cov * 8
        print(f'  Conf >= {threshold:.0%}: {avg_count:.1f}/8 fixtures ({avg_cov:.1%} coverage)')
else:
    print('Fixture-level backtest requires home_team/away_team columns in dataset')
    print('Legacy dataset detected — showing per-prediction accuracy instead')
    
    split_idx = int(len(df) * 0.80)
    df_test = df.iloc[split_idx:].copy()
    
    for market_name, model_data in models.items():
        target = model_data['target']
        if target not in df_test.columns:
            continue
        mask = df_test[target].notna() & df_test[FEATURES].notna().all(axis=1)
        X_t = df_test.loc[mask, FEATURES].fillna(0).values
        y_t = df_test.loc[mask, target].values.astype(int)
        
        xgb_p = model_data['xgb'].predict_proba(X_t)[:, 1]
        lgb_p = model_data['lgb'].predict_proba(X_t)[:, 1]
        ens_p = model_data['iso'].transform((xgb_p + lgb_p) / 2)
        
        # High confidence subset
        for threshold in [0.70, 0.75, 0.80, 0.85, 0.90]:
            mask_hc = ens_p >= threshold
            if mask_hc.sum() > 20:
                acc = y_t[mask_hc].mean()
                print(f'  {market_name:12s} @ conf>={threshold:.0%}: n={mask_hc.sum():4d} acc={acc:.1%}')


In [ ]:
# ── 8. FEATURE IMPORTANCE ANALYSIS ───────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#1a1a2e'
matplotlib.rcParams['text.color'] = 'white'
matplotlib.rcParams['axes.facecolor'] = '#16213e'
matplotlib.rcParams['axes.edgecolor'] = '#0f3460'
matplotlib.rcParams['xtick.color'] = 'white'
matplotlib.rcParams['ytick.color'] = 'white'

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('VFL Empire — Feature Importance per Market', color='#e94560', fontsize=16, fontweight='bold')

market_order = ['over_15', 'under_35', 'gg', 'home_win', 'draw', 'away_win']
color_map = {'over_15': '#00d4ff', 'under_35': '#ff6b6b', 'gg': '#ffd700',
             'home_win': '#51cf66', 'draw': '#fd7e14', 'away_win': '#cc5de8'}

for idx, market_name in enumerate(market_order):
    if market_name not in models:
        continue
    ax = axes[idx // 3][idx % 3]
    model_data = models[market_name]
    
    # XGBoost importance
    xgb_imp = model_data['xgb'].feature_importances_
    top_n = 10
    top_idx = np.argsort(xgb_imp)[-top_n:]
    
    bars = ax.barh(
        [FEATURES[i] for i in top_idx],
        [xgb_imp[i] for i in top_idx],
        color=color_map.get(market_name, '#adb5bd')
    )
    ax.set_title(market_name.replace('_', ' ').upper(), color='white', fontsize=11, fontweight='bold')
    val_acc = model_data['val_acc']
    ax.set_xlabel(f'Importance | Val Acc: {val_acc:.1%}', color='#adb5bd', fontsize=9)
    ax.tick_params(labelsize=8)

plt.tight_layout()
save_path = '/content/feature_importance.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
from pathlib import Path
import shutil
shutil.copy(save_path, str(MODELS_DIR / 'feature_importance.png'))
print(f'Feature importance chart saved.')
plt.show()


In [ ]:
# ── 9. CALIBRATION CURVES ─────────────────────────────────────────────────────
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('VFL Empire — Calibration Curves (Predicted vs Actual)', color='#e94560', fontsize=14, fontweight='bold')

split_idx = int(len(df) * 0.80)
df_val = df.iloc[split_idx:].copy()

for idx, market_name in enumerate(market_order):
    if market_name not in models:
        continue
    model_data = models[market_name]
    target = model_data['target']
    if target not in df_val.columns:
        continue
    
    ax = axes[idx // 3][idx % 3]
    mask = df_val[target].notna() & df_val[FEATURES].notna().all(axis=1)
    X_v = df_val.loc[mask, FEATURES].fillna(0).values
    y_v = df_val.loc[mask, target].values.astype(int)
    
    if len(X_v) < 100:
        continue
    
    xgb_p = model_data['xgb'].predict_proba(X_v)[:, 1]
    lgb_p = model_data['lgb'].predict_proba(X_v)[:, 1]
    raw_p = (xgb_p + lgb_p) / 2
    cal_p = model_data['iso'].transform(raw_p)
    
    # Calibration curves
    frac_pos_raw, mean_pred_raw = calibration_curve(y_v, raw_p, n_bins=8)
    frac_pos_cal, mean_pred_cal = calibration_curve(y_v, cal_p, n_bins=8)
    
    ax.plot([0, 1], [0, 1], 'w--', alpha=0.5, label='Perfect')
    ax.plot(mean_pred_raw, frac_pos_raw, 'o-', color='#ff6b6b', label='Raw ensemble', linewidth=2)
    ax.plot(mean_pred_cal, frac_pos_cal, 's-', color='#51cf66', label='Calibrated', linewidth=2)
    ax.set_title(market_name.replace('_', ' ').upper(), color='white')
    ax.set_xlabel('Predicted probability', color='#adb5bd')
    ax.set_ylabel('Actual win rate', color='#adb5bd')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2, color='white')

plt.tight_layout()
save_path = '/content/calibration_curves.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
import shutil
shutil.copy(save_path, str(MODELS_DIR / 'calibration_curves.png'))
print('Calibration curves saved.')
plt.show()


## ✅ Training Complete!

All models are saved to **Google Drive** at `vfl_empire_models/`.

**Files saved:**
- `xgb_over_15.json`, `lgb_over_15.txt`, `iso_over_15.pkl`
- `xgb_under_35.json`, `lgb_under_35.txt`, `iso_under_35.pkl`
- `xgb_gg.json`, `lgb_gg.txt`, `iso_gg.pkl`
- `xgb_home_win.json` etc.
- `model_features.json` — feature list
- `training_summary_rich.json` — accuracy metrics

**Next steps (automated on VM):**
1. `colab-cli pull-nb` pulls this notebook back to VM
2. `vfl_fixture_predictor.py` loads models from Drive path
3. All 8 fixtures predicted per matchday with calibrated probabilities
